In [ ]:
# Run in conda ibiz environment
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import pyproj
from scipy.spatial import cKDTree

# import importlib
# import preprocess_radargrams_configs
# importlib.reload(preprocess_radargrams_configs)

In [ ]:
# location of tar file
# NOTE: Start with 2016 data, as it is more recent and has better coverage
from preprocess_radargrams_configs import PATH_2016_AN_UTIG_ER2HI1B, PATH_2016_AN_UTIG_ER2HI2, CODE_1B, CODE_2, ROWS_CLIPPED_BLANKING
from preprocess_radargrams_configs import AMPLITUDE_VMIN, AMPLITUDE_VMAX, COLOR_MAP

PATH_2016_AN_UTIG_ER2HI1B_folder = os.path.join(PATH_2016_AN_UTIG_ER2HI1B, "2016_AN_UTIG.ER2HI1B")

# Extract the list of .nc files in the folder
list_of_nc_files_2016_UTIG = [
    f for f in os.listdir(PATH_2016_AN_UTIG_ER2HI1B_folder) 
    if f.endswith(".nc")
    ]

list_of_nc_files_2016_UTIG_full_paths = [
    os.path.join(PATH_2016_AN_UTIG_ER2HI1B_folder, f)
    for f in os.listdir(PATH_2016_AN_UTIG_ER2HI1B_folder)
    if f.endswith(".nc")
]

# Proprocessing pipeline for radar data

## Steps
- Start from one image first
    - **high gain** more relevant for bed detection, **low gain** more relevant for surface detection
- Extract location of bed 
- crop IBIZ 
- Have a fixed scale
- Combine with bed picks

# Acronyms
- ER2HI1B - ER: EAGLE Radar (?!) - HI: HiCARS - 1B: Level 1B data
- ER2HI2 - ER: EAGLE Radar (?!) - HI: HiCARS - 2: Level 2 data
    - 2016 refers to the 2016/17 season so Jan 2017 is part of it
    - files are considered granules

In [ ]:
path_to_a_radargram = list_of_nc_files_2016_UTIG_full_paths[0]

radargram_ds = xr.open_dataset(path_to_a_radargram)
print("Granule ID:", radargram_ds.granule_id)
print("Radargram shape:", radargram_ds["amplitude_high_gain"].shape)
radargram_ds

In [ ]:
fig, ax = plt.subplots(figsize = (12, 6))

pcm = ax.pcolormesh(
    # pcolormesh expects input C to be (nrows, ncols) so we have to transpose the data array
    # 400 top rows radargram_ds["amplitude_high_gain"].T[0:400, :],
    radargram_ds["amplitude_high_gain"].T[:, :],
    cmap = "gray_r"
)

# radar convention: surface at top
# Now it is visualised like a table with the origin in the TOP LEFT corner
ax.invert_yaxis() 
plt.colorbar(pcm, ax = ax, label = "Amplitude")
plt.show()

# Find matching L2 data (bed picks)

Granule ID: ER2HI1B_2017024_TOT1_JKB2r_X19a_000

## Construct full path from info we already have 

In [ ]:
radargram_grandule_id_L2 = radargram_ds.granule_id[:5] + "2" + radargram_ds.granule_id[7:]
radargram_grandule_id_L2_cropped = radargram_grandule_id_L2[:-3]

PATH_L1B = PATH_2016_AN_UTIG_ER2HI2 + "/" + radargram_grandule_id_L2_cropped + "icethk.txt"
print(PATH_L1B)
# print("/Users/kim.bente/Documents/ANT_QGIS/AADC-DATA/AAS_4346_EAGLE_ICECAP_LEVEL2_AEROGEOPHYSICS/EAGLE-2015-2016-Level2/Level2/2016_AN_UTIG.ER2HI2/ER2HI2_2017024_TOT1_JKB2r_X19a_icethk.txt")

print()
# print the header lines from the L1B text file
with open(PATH_L1B, "r") as f:
    header_lines = [line for line in f if line.startswith("#")]

for line in header_lines:
    print(line, end = "")

In [ ]:
# grab last line for the column headers
col_names = header_lines[-1].lstrip("#").split()
print(col_names)

# now import the data into a pandas dataframe, skipping the header lines, but adding the column names from the last header line
l2_df = pd.read_csv(
    PATH_L1B,
    sep = r"\s+", 
    # skips any line starting with '#'
    comment = "#", 
    header = None, 
    names = col_names,
)

l2_df.head()

In [ ]:
# summary statistics
l2_df.describe()

## Convert to Polar Stereographic

- to calculate distance

In [ ]:
# This is how projections work. check through tool
lonlat_to_polarstereo = pyproj.Transformer.from_crs(crs_from = pyproj.CRS("epsg:4326"),
                                                    crs_to = pyproj.CRS("epsg:3031"),
                                                    always_xy = True) # xy order convention

# Pass lon and lat columns, returns x and y columns in polar stereographic coordinates
x_array, y_array = lonlat_to_polarstereo.transform(l2_df["LON"], l2_df["LAT"])

l2_df["X"] = x_array
l2_df["Y"] = y_array

# Calculate the distance between each point and the next point in the dataframe, using the X and Y columns
distance_to_next = ((l2_df["X"].shift(-1) - l2_df["X"])**2 + (l2_df["Y"].shift(-1) - l2_df["Y"])**2)**0.5
print("Distance to next point in meters:")
print(distance_to_next.describe())

# Join

Join additional info from L2 onto L1B. This may include:
- bed elevation (subtract bed elevation + flight altitude from L1B to get bed elevation)
    - unclear about surface range
- PARTIAL_BED_REFLECT # Field 10: Bed reflection coefficient @ 60 MHz (dB wrt perfect reflector; NO ICE LOSS ACCOUNTING)
- SRF_REFLECT # Field 11: Surface reflection coefficient @ 60 MHz (dB wrt perfect reflector)

Future:
- Use YEAR and DOY as alternative or additional way to match

# Match via LON LAT

- L1B data is float32
- L2 data is float64. When we simply convert L2 to float32 we match rather well. These are still some numerical issues but a tree and generate a 1:1 match.

In [ ]:
# Overlap of L2 coordinates and radargram coordinates
fig, ax = plt.subplots(figsize = (8, 8))


ax.scatter(l2_df["LON"].values, l2_df["LAT"].values, 
           s = 9, alpha = 0.1, label = "L2 file", c = l2_df.index, cmap = "viridis")
ax.scatter(radargram_ds["lon"].values, radargram_ds["lat"].values, 
           s = 1, alpha = 0.1, label = "radargram (L1B)", c = np.arange(radargram_ds.time.size), cmap = "YlOrRd_r")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.set_title("Radargram vs L2 coordinates")
plt.show()

# Join

In [ ]:
# Use consistent dtype (float64) for both — don't force float32
radar_lon = radargram_ds["lon"].values.astype(np.float64)
radar_lat = radargram_ds["lat"].values.astype(np.float64)

l2_lon = l2_df["LON"].values.astype(np.float64)
l2_lat = l2_df["LAT"].values.astype(np.float64)

radar_coords = np.column_stack([radar_lon, radar_lat])
l2_coords = np.column_stack([l2_lon, l2_lat])

# Build tree on l2_df (the side we're pulling matches from)
tree = cKDTree(l2_coords)
tol = 1e-5  # degrees; tune to your data's actual precision/spacing

distances, indices = tree.query(radar_coords, distance_upper_bound = tol)

# distances == inf / indices == len(l2_coords) means "no match found"
has_match = distances < tol

# indices is len(radargram_ds) long, aligned to radar order
# invalid indices point out-of-bounds, so clip/mask them before indexing l2_df
safe_indices = np.where(has_match, indices, 0)  # placeholder for unmatched rows

matched_l2_subset = l2_df.iloc[safe_indices].reset_index(drop=True)

# Null out rows that didn't actually match
matched_l2_subset.loc[~has_match] = np.nan

print(f"Matched: {has_match.sum()} / {len(radar_coords)}")

# Select columns I want to transfer
selected_cols = ["THK", "BED_ELEVATION", "SURFACE_ELEVATION", "PARTIAL_BED_REFLECT", "SRF_REFLECT", "SRF_RNG"]

for col in selected_cols:
    # along time dim
    radargram_ds[f"l2_{col}"] = ("time", matched_l2_subset[col].values)

# free up memory
del tree

## Process L1B - Convert fasttime to meters

- Fasttime 3200 x Time 4000
- fasttime is 2-way travel time in µseconds
- Conversion differs for air and ice so do a dual approach
- every column will consequently have a sightly different converion of fasttime

In [ ]:
print("Fasttime units:")
print(radargram_ds.fasttime.long_name)
print()
print("Amplitude high gain units:")
print(radargram_ds.amplitude_high_gain.units)
print()
print(radargram_ds.amplitude_high_gain)
 # counts in dB

In [ ]:
# speed of light in vacuum
c = 2.998e8  # m/s

# relative permittivity (epsilon_r) of ice (~ 3.15 is standard for glacial ice)
epsilon_ice = 3.15

# refractive index of ice
n = np.sqrt(epsilon_ice)

# speed of light in ice
v_ice = c / n 

# --- Air gap handling ---
# "free air content" = one-way distance from antenna to surface, in meters
air_gap_m = radargram_ds.altitude.values - radargram_ds.l2_SURFACE_ELEVATION.values  # plane altitude above surface

# Two-way travel time spent in air (round trip through air only, given air gap in meters and speed of light in vacuum)
# Maybe 10% of time air travel
air_twtt_seconds = (2 * air_gap_m) / c

# Convert fasttime (TWTT, in microseconds) to seconds
fasttime_in_seconds = radargram_ds.fasttime.values * 1e-6

# Subtract the air portion, leaving only the ice portion of the TWTT
# fasttime_in_seconds is a column vector (y-axis), air_twtt_seconds is a row vector (x-axis), so we can broadcast the subtraction to get a 2D array of ice TWTT values
# NOTE: negative values mean time still in air, so we can mask those out later if needed
ice_twtt_seconds = fasttime_in_seconds.ravel()[:, np.newaxis] - air_twtt_seconds.ravel()[np.newaxis, :]

# --- Mask: True where sample is still in air (before surface return) ---
in_air_mask = ice_twtt_seconds < 0

# Container
one_way_distance_m = np.empty_like(ice_twtt_seconds)

# Fill air bins only: one way distance from altitude
one_way_distance_m[in_air_mask] = c * fasttime_in_seconds[:, np.newaxis].repeat(ice_twtt_seconds.shape[1], axis = 1)[in_air_mask] / 2

# Fill ice bins only
# Note: Maybe use cell atop
one_way_distance_m[~ in_air_mask] = (air_gap_m[np.newaxis, :] + v_ice * ice_twtt_seconds / 2)[~ in_air_mask]

# Get elevation above sea level (m) for each sample in the radargram, by subtracting the one-way distance from the plane altitude
elevation_m = radargram_ds.altitude.values[None, :] - one_way_distance_m

In [ ]:
elevation_m
# radargram_ds["amplitude_high_gain"].values[-10:-1, 0:10]

In [ ]:
# Consistency check:
residual = radargram_ds["altitude"] - radargram_ds["l2_SRF_RNG"] - radargram_ds["l2_SURFACE_ELEVATION"]
np.max(np.abs(residual))

In [ ]:
print(ROWS_CLIPPED_BLANKING)
# 75 or 76

ROWS_CLIPPED_BLANKING = 120

in_air_mask_shifted = np.pad(
    in_air_mask,
    pad_width = ((ROWS_CLIPPED_BLANKING, 0), (0, 0)),  # pad ROWS_CLIPPED_BLANKING rows at top, 0 at bottom
    mode = "constant",
    constant_values = False
)[:-ROWS_CLIPPED_BLANKING, :]

print("in_air_mask shape:", in_air_mask.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# amplitude is (time, fasttime_dim) -> transpose to (fasttime_dim, time) for pcolormesh's (Y, X) convention
# now shape (3200, 4000)
amplitude = radargram_ds["amplitude_high_gain"].values.T 
# shape (3200,)
fasttime = radargram_ds.fasttime.values                   
trace_idx = np.arange(amplitude.shape[1])          

mesh = ax.pcolormesh(trace_idx, fasttime, amplitude, shading = "auto", cmap = "gray_r")
fig.colorbar(mesh, ax = ax, label = "Amplitude (high gain, dB)")

# Overlay mask — also needs the same transpose, since in_air_mask was built as (n_fasttime, n_traces)
mask_overlay = np.where(in_air_mask_shifted, 1, np.nan)  # in_air_mask should already be (fasttime, time) shaped
ax.pcolormesh(trace_idx, fasttime, mask_overlay, shading = "auto", cmap = "Reds", alpha = 0.3, vmin= 0, vmax = 1)

ax.set_xlabel("Trace index (along track)")
ax.set_ylabel("Fasttime (µs)")
ax.set_title("Radargram with air-region mask overlay")
ax.invert_yaxis()

# ax.set_ylim(8, 10)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# amplitude is (time, fasttime_dim) -> transpose to (fasttime_dim, time) for pcolormesh's (Y, X) convention
# now shape (3200, 4000)
amplitude = radargram_ds["amplitude_high_gain"].values.T 
# shape (3200,)
fasttime = np.arange(radargram_ds.fasttime.values.shape[0])                  
trace_idx = np.arange(amplitude.shape[1])          

mesh = ax.pcolormesh(trace_idx, fasttime, amplitude, shading = "auto", cmap = "gray_r")
fig.colorbar(mesh, ax = ax, label = "Amplitude (high gain, dB)")

# Overlay mask — also needs the same transpose, since in_air_mask was built as (n_fasttime, n_traces)
mask_overlay = np.where(in_air_mask_shifted, 1, np.nan)  # in_air_mask should already be (fasttime, time) shaped
ax.pcolormesh(trace_idx, fasttime, mask_overlay, shading = "auto", cmap = "Reds", alpha = 0.3, vmin= 0, vmax = 1)

ax.set_xlabel("Trace index (along track)")
ax.set_ylabel("Fasttime (µs)")
ax.set_title("Radargram with air-region mask overlay")
ax.invert_yaxis()

# ax.set_ylim(8, 10)

plt.tight_layout()
plt.show()

In [ ]:
print("in_air_mask shape:", in_air_mask.shape)
print("amplitude shape:", amplitude.shape)
print("Number of True (in-air) cells:", in_air_mask.sum())
print("Total cells:", in_air_mask.size)
print("Fraction in air:", in_air_mask.sum() / in_air_mask.size)

In [ ]:
fasttime_us = radargram_ds.fasttime.values  # shape (n_fasttime,)
dt_us = fasttime_us[1] - fasttime_us[0]      # sample spacing in microseconds
dt_s = dt_us * 1e-6

# Expected row index of the surface return, per trace
expected_surface_row = air_twtt_seconds / dt_s   # air_twtt_seconds already in seconds, shape (n_traces,)

In [ ]:
expected_surface_row

In [ ]:
amp = radargram_ds.amplitude_low_gain.values.T[350:, :]  # or whatever your power/amplitude variable is called

# crude but effective: row of max amplitude per trace = surface return
observed_surface_row = np.argmax(amp, axis = 0)  # shape (n_traces,)

In [ ]:
observed_surface_row

In [ ]:
350+121-343

# Questions:
How close are the picks? 
Can we match them with the radar data purely based in lon lat?
use reflectivity and other for correlation analysis (after) or also for processing?
Aircraft height is not in the L2 dataset.

Off-set to fix aircraft altitude...

Output:
256 pixels x 256 picels
256 x 20 m = 5 km around


In [ ]:
bed_elevation = radargram_ds.l2_BED_ELEVATION.values  # shape (4000,)

# For each column (trace), find the row index where elevation_m is closest to bed_elevation
bed_idx = np.argmin(np.abs(elevation_m - bed_elevation[np.newaxis, :]), axis = 0)
bed_idx = bed_idx + 130

print("bed_idx shape:", bed_idx.shape)  # (4000,) — one index per trace

trace_cols = np.arange(elevation_m.shape[1])

bed_mask = np.zeros_like(elevation_m, dtype=bool)
bed_mask[bed_idx, trace_cols] = True

In [ ]:
fig, ax = plt.subplots(figsize = (12, 6))

# amplitude is (time, fasttime_dim) -> transpose to (fasttime_dim, time) for pcolormesh's (Y, X) convention
amplitude = radargram_ds["amplitude_high_gain"].values.T 
# y axis
fasttime_idx = np.arange(radargram_ds.fasttime.values.shape[0])                 
# x axis
trace_idx = np.arange(amplitude.shape[1])               

# RADARGRAM BASE
mesh = ax.pcolormesh(trace_idx, fasttime_idx, amplitude, shading = "auto", cmap = "gray_r", vmin = 110, vmax = 215)
fig.colorbar(mesh, ax = ax, label = "Amplitude (high gain, dB)")

# Overlay mask — also needs the same transpose, since in_air_mask was built as (n_fasttime, n_traces)
# Turn 0's into NaNs for transparency, and 1's into 1's for red overlay
mask_overlay = np.where(bed_mask, 1, np.nan) 
ax.pcolormesh(trace_idx, fasttime_idx, mask_overlay, shading = "auto", cmap = "Reds", alpha = 0.9, vmin = 0, vmax = 1)

ax.set_xlabel("Trace index (along track)")
ax.set_ylabel("Fasttime (µs)")
ax.set_title("Radargram with air-region mask overlay")

ax.set_ylim(1000, 1500)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
np.argsort(radargram_ds.lat.values)[:3]


In [ ]:
print(radargram_ds.lat.values[0:3])
print(radargram_ds.lon.values[0:3])

In [ ]:
x_center_index = 650

HALFWIDTH = 200
x_min_index = x_center_index - HALFWIDTH
x_max_index = x_center_index + HALFWIDTH

# look up bed
y_center_index = bed_idx[x_center_index].item()
y_min_index = y_center_index - HALFWIDTH + ROWS_CLIPPED_BLANKING 
y_max_index = y_center_index + HALFWIDTH + ROWS_CLIPPED_BLANKING 


In [ ]:
tile = amplitude[y_min_index:y_max_index, x_min_index:x_max_index]
# tile = amplitude[:, x_min_index:x_max_index]

fig, ax = plt.subplots(figsize = (8, 6))
mesh = ax.pcolormesh(tile, shading = "auto", cmap = "gray_r")
fig.colorbar(mesh, ax = ax, label = "Amplitude (dB)")
ax.invert_yaxis()
plt.show()

In [ ]:
radargam_ds["l2_SURFACE_ELEVATION"]

# METADATA

- vertical span
- horizontal span

